---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# Deploy Agent for Local Inference

## Deploy as code

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow
from mlflow.entities import SpanType

load_dotenv()

openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-2.5-flash-lite"
EXPERIMENT_NAME = os.environ.get("EXPERIMENT_4_NAME", "mlflow-agent-4")

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

experiment = mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.openai.autolog()

/Users/jon/Documents/GitHub/practical-agent-ops-mlflow3/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Create active agent model
mlflow.set_active_model(name="mlflow-agent-v1")

2026/04/25 11:39:02 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-113c39a89eed48bdb7e3553e5fa53c99


LoggedModel()

## Log Model

Once the active model is set, we then are going to log the code model as an MLflow model

In [3]:
with mlflow.start_run():
    info = mlflow.pyfunc.log_model(
        name="mlflow_agent",
        python_model="mlflow_agent.py",
        pip_requirements=["openai", "mlflow"],
    )

/Users/jon/Documents/GitHub/practical-agent-ops-mlflow3/training/agents/experiment4/mlflow_agent.py:34: FutureWarning: ``mlflow.pyfunc.model.ChatModel`` is deprecated since 3.0.0. This method will be removed in a future release. Use ``ResponsesAgent`` instead.
  AGENT = MLflowAgent()
2026/04/25 11:39:09 WARNING mlflow.pyfunc: Default values for temperature, n and stream in ChatParams will be removed in the next release. Specify them in the input example explicitly if needed.
2026/04/25 11:39:09 INFO mlflow.pyfunc: Predicting on input example to validate output
2026/04/25 11:39:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/var/folders/hp/w76clljs12z5x2b15j1n42tr0000gn/T/tmpjodp_7ub/model/mlflow_agent.py:34: FutureWarning: ``mlflow.pyfunc.model.ChatModel`` is deprecated since 3.0.0. This method will be removed in a future release. Use ``ResponsesAgent`` instead.
  AGENT = ML

🏃 View run rumbling-hare-643 at: http://127.0.0.1:5000/#/experiments/9/runs/885d232d9d504cac921bcbac24458d42
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/9


Now that we've logged the model successfuly, we need to submit it to the 'Model Registry' in order to deploy it to a local server. We register the agent with the classic `mlflow.register_model` function that we're used to from traditional ML.

*We can also perform this action from the UI*

In [ ]:
mlflow.register_model(model_uri=info.model_uri, name="mlflow_agent")

## A little about MLflow client

In [ ]:
# Check the model is registerd
from mlflow import MlflowClient

client = MlflowClient(tracking_uri="http://127.0.0.1:5000")

# List all registered models
for rm in client.search_registered_models():
    print(rm.name)
    for v in rm.latest_versions:
        print(f"  version {v.version}  (run_id={v.run_id}, status={v.status})")
        print(f"  models:/{rm.name}/{v.version}")

OpenAI Completions Call
  version 2  (run_id=, status=READY)
  models:/OpenAI Completions Call/2
mlflow-assistant
  version 2  (run_id=885d232d9d504cac921bcbac24458d42, status=READY)
  models:/mlflow-assistant/2


In [14]:
mv = client.get_model_version(name="mlflow-assistant", version=2)

In [15]:
mv

<ModelVersion: aliases=[], creation_timestamp=1777131594977, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1777131594977, metrics=None, model_id=None, name='mlflow-assistant', params=None, run_id='885d232d9d504cac921bcbac24458d42', run_link='', source='models:/m-af8ef6bdf6b84a12951194867e763b13', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>

In [ ]:
mlflow models serve -m models:/mlflow-assistant/2 -p 5678 --env-manager=local

In [3]:
# Perform model inference
import json
import requests

payload = json.dumps({
    "inputs": {"messages": [{"role": "user", "content": "Tell a joke!"}]},
    "params": {
        "temperature": 0.5,
        "max_tokens": 20,
    },
})
response = requests.post(
    url=f"http://127.0.0.1:5678/invocations",
    data=payload,
    headers={"Content-Type": "application/json"},
)
print(response.json())

{'predictions': {'choices': [{'message': {'role': 'assistant', 'content': "Why don't scientists trust atoms?\n\nBecause they make up everything!"}, 'index': 0, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 5, 'completion_tokens': 15, 'total_tokens': 20}, 'id': 'AevsaY3WMbi3_uMPlNnHkA8', 'model': 'gemini-2.5-flash-lite', 'object': 'chat.completion', 'created': 1777134337}}


In [ ]:
# If you need to stop the server, run in terminal:
kill -9 $(lsof -t -i:5678)